In [1]:
import duckdb
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show
from bokeh.layouts import gridplot
from bokeh.models import Span
from bokeh.io import output_notebook, curdoc

output_notebook()
curdoc().theme = 'dark_minimal'

Loading BokehJS ...

---
### Chargement des données 
(DuckDb File)

In [2]:
DB_PATH = 'crypto.db'
conn = duckdb.connect(DB_PATH, read_only=True)

SYMBOL = 'ETHUSDT' #BTC ETH AAVE dispo
INTERVAL = '1h' #1h only 
START_DATE = '2022-01-01'
END_DATE = '2025-12-01'

SMA_PERIOD = 10

In [3]:
spot_df = conn.execute("""
    SELECT open_time as timestamp, close as spot_close
    FROM spot
    WHERE exchange = 'binance' AND symbol = ? AND interval = ?
    AND open_time >= ? AND open_time < ?
    ORDER BY open_time
""", [SYMBOL, INTERVAL, START_DATE, END_DATE]).df()

futures_df = conn.execute("""
    SELECT open_time as timestamp, close as futures_close
    FROM futures
    WHERE exchange = 'binance' AND symbol = ? AND interval = ?
    AND open_time >= ? AND open_time < ?
    ORDER BY open_time
""", [SYMBOL, INTERVAL, START_DATE, END_DATE]).df()

df = pd.merge(spot_df, futures_df, on='timestamp', how='inner')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp')

In [4]:
df['basis'] = ((df['futures_close'] - df['spot_close']) / df['spot_close']) * 100

In [5]:
df.head()

,spot_close,futures_close,basis
timestamp,,,
2022-01-01 00:00:00,3723.04,3721.67,-0.036798
2022-01-01 01:00:00,3724.89,3723.71,-0.031679
2022-01-01 02:00:00,3728.32,3727.03,-0.034600
2022-01-01 03:00:00,3723.96,3722.07,-0.050752
2022-01-01 04:00:00,3708.21,3707.30,-0.024540


---

In [6]:
funding_df = conn.execute("""
    SELECT DATE_TRUNC('hour', timestamp) as timestamp, funding_rate
    FROM funding_rates
    WHERE exchange = 'binance' AND symbol = ?
    ORDER BY timestamp
""", [SYMBOL]).df()

if len(funding_df) > 0:
    funding_df['timestamp'] = pd.to_datetime(funding_df['timestamp'])
    funding_df = funding_df.set_index('timestamp')
    df = df.join(funding_df, how='left')
    df['funding_rate'] = df['funding_rate'].ffill() * 100 # convert to percentage
else:
    df['funding_rate'] = np.nan

In [7]:
df.head()

,spot_close,futures_close,basis,funding_rate
timestamp,,,,
2022-01-01 00:00:00,3723.04,3721.67,-0.036798,0.01
2022-01-01 01:00:00,3724.89,3723.71,-0.031679,0.01
2022-01-01 02:00:00,3728.32,3727.03,-0.034600,0.01
2022-01-01 03:00:00,3723.96,3722.07,-0.050752,0.01
2022-01-01 04:00:00,3708.21,3707.30,-0.024540,0.01


---
Affichage

In [8]:
# Layout
PLOT_WIDTH = 1400
TOTAL_HEIGHT = 800
HEIGHT_RATIOS = (0.5, 0.25, 0.25)  # price, basis, funding

In [9]:
# Calculate SMAs
df['basis_sma'] = df['basis'].rolling(SMA_PERIOD).mean()
df['funding_sma'] = df['funding_rate'].rolling(SMA_PERIOD).mean()

# Calculate heights from ratios
h_price = int(TOTAL_HEIGHT * HEIGHT_RATIOS[0])
h_basis = int(TOTAL_HEIGHT * HEIGHT_RATIOS[1])
h_funding = int(TOTAL_HEIGHT * HEIGHT_RATIOS[2])

# Plot 1: Price (Futures + Spot)
p1 = figure(
    title=f'{SYMBOL} Price',
    x_axis_type='datetime',
    width=PLOT_WIDTH,
    height=h_price,
    background_fill_color='#1a1a1a',
    border_fill_color='#1a1a1a'
)
p1.line(df.index, df['futures_close'], color='#00BFFF', line_width=1, legend_label='Futures')
p1.line(df.index, df['spot_close'], color='#FF6347', line_width=1, legend_label='Spot')
p1.yaxis.axis_label = 'Price (USDT)'
p1.legend.click_policy = "hide"

# Plot 2: BASIS
p2 = figure(
    title='BASIS (%)',
    x_axis_type='datetime',
    x_range=p1.x_range,
    width=PLOT_WIDTH,
    height=h_basis,
    background_fill_color='#1a1a1a',
    border_fill_color='#1a1a1a'
)
p2.line(df.index, df['basis'], color='#FFD700', line_width=1, legend_label='Basis')
p2.line(df.index, df['basis_sma'], color='white', line_width=1.5, legend_label=f'SMA({SMA_PERIOD})')
p2.add_layout(Span(location=0, dimension='width', line_color='white', line_dash='dashed', line_alpha=0.5))
p2.yaxis.axis_label = 'BASIS (%)'
p2.legend.click_policy = "hide"

# Plot 3: Funding Rate
p3 = figure(
    title='Funding Rate (%)',
    x_axis_type='datetime',
    x_range=p1.x_range,
    width=PLOT_WIDTH,
    height=h_funding,
    background_fill_color='#1a1a1a',
    border_fill_color='#1a1a1a'
)
p3.line(df.index, df['funding_rate'], color='#FF69B4', line_width=1, legend_label='Funding')
p3.line(df.index, df['funding_sma'], color='white', line_width=1.5, legend_label=f'SMA({SMA_PERIOD})')
p3.add_layout(Span(location=0.01, dimension='width', line_color='white', line_dash='dashed', line_alpha=0.5))
p3.yaxis.axis_label = 'Funding Rate (%)'
p3.legend.click_policy = "hide"

grid = gridplot([[p1], [p2], [p3]], merge_tools=True)
show(grid)

---
Trend Indicateur

In [10]:
# Price colored by basis sign
df['color'] = np.where(df['basis'] >= 0, '#00FF00', '#FF0000')

xs, ys, colors = [], [], []
for i in range(len(df) - 1):
    xs.append([df.index[i], df.index[i+1]])
    ys.append([df['futures_close'].iloc[i], df['futures_close'].iloc[i+1]])
    colors.append(df['color'].iloc[i])

p4 = figure(
    title=f'{SYMBOL} Price (Green: Basis > 0, Red: Basis < 0)',
    x_axis_type='datetime',
    width=1400,
    height=400,
    background_fill_color='#1a1a1a',
    border_fill_color='#1a1a1a'
)
p4.multi_line(xs, ys, line_color=colors, line_width=1)
p4.yaxis.axis_label = 'Price (USDT)'

show(p4)

---
Indicateur De Sentiment

In [11]:
# Panic/Euphoria detection
ZSCORE_WINDOW = 100  # 7 jours
ZSCORE_THRESHOLD = 2  # < -2 panic, > +2 euphori

In [12]:
from bokeh.models import BoxAnnotation

# Z-Score calculation
rolling_mean = df['basis'].rolling(ZSCORE_WINDOW).mean()
rolling_std = df['basis'].rolling(ZSCORE_WINDOW).std()
df['basis_zscore'] = (df['basis'] - rolling_mean) / rolling_std

# Detect panic (< -threshold) and euphoria (> +threshold)
df['panic'] = df['basis_zscore'] < -ZSCORE_THRESHOLD
df['euphoria'] = df['basis_zscore'] > ZSCORE_THRESHOLD

# Heights
h_indicator = 200

# Plot 1: Price with panic/euphoria zones
p_price = figure(
    title=f'{SYMBOL} Price - Panic (red) & Euphoria (green)',
    x_axis_type='datetime',
    width=PLOT_WIDTH,
    height=h_price,
    background_fill_color='#1a1a1a',
    border_fill_color='#1a1a1a'
)
p_price.line(df.index, df['futures_close'], color='#00BFFF', line_width=1)
p_price.yaxis.axis_label = 'Price (USDT)'

# Panic zones (red)
for idx in df[df['panic']].index:
    p_price.add_layout(BoxAnnotation(left=idx, right=idx + pd.Timedelta(hours=1),
                                      fill_color='red', fill_alpha=0.3))

# Euphoria zones (green)
for idx in df[df['euphoria']].index:
    p_price.add_layout(BoxAnnotation(left=idx, right=idx + pd.Timedelta(hours=1),
                                      fill_color='lime', fill_alpha=0.3))

# Plot 2: Basis
p_basis = figure(
    title='Basis (%)',
    x_axis_type='datetime',
    x_range=p_price.x_range,
    width=PLOT_WIDTH,
    height=h_indicator,
    background_fill_color='#1a1a1a',
    border_fill_color='#1a1a1a'
)
p_basis.line(df.index, df['basis'], color='#FFD700', line_width=1)
p_basis.add_layout(Span(location=0, dimension='width', line_color='white', line_dash='dashed', line_alpha=0.5))
p_basis.yaxis.axis_label = 'Basis (%)'

# Plot 3: Z-Score with thresholds
p_zscore = figure(
    title=f'Basis Z-Score (±{ZSCORE_THRESHOLD})',
    x_axis_type='datetime',
    x_range=p_price.x_range,
    width=PLOT_WIDTH,
    height=h_indicator,
    background_fill_color='#1a1a1a',
    border_fill_color='#1a1a1a'
)
p_zscore.line(df.index, df['basis_zscore'], color='white', line_width=1)
p_zscore.add_layout(Span(location=-ZSCORE_THRESHOLD, dimension='width', line_color='red', line_dash='dashed', line_width=2))
p_zscore.add_layout(Span(location=ZSCORE_THRESHOLD, dimension='width', line_color='lime', line_dash='dashed', line_width=2))
p_zscore.add_layout(Span(location=0, dimension='width', line_color='white', line_dash='dashed', line_alpha=0.3))
p_zscore.yaxis.axis_label = 'Z-Score'

grid = gridplot([[p_price], [p_basis], [p_zscore]], merge_tools=True)
show(grid)

print(f"Panic events (Z < -{ZSCORE_THRESHOLD}): {df['panic'].sum()}")
print(f"Euphoria events (Z > +{ZSCORE_THRESHOLD}): {df['euphoria'].sum()}")

Panic events (Z < -2): 927
Euphoria events (Z > +2): 1109
